# 6장 구조화 출력 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers torch (예제 6-2만 씁니다)
# 6장과 관통 프로젝트 2단계에서 씁니다
import csv
import random

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 열: 번호, 문의, 라벨, 긴급, 비고
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). 5장과 같습니다(Anthropic API의 기본 모델, 2026-11 확인)
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


def parsed(text):
    """응답 텍스트를 JSON으로 파싱합니다. 실패하면 "파싱 오류"와 응답 앞부분을 돌려줍니다"""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return f'파싱 오류: {text.strip()[:40]!r}'


def load_local_model():
    """예제 6-2용. 1장의 공개 LLM을 PC에서 불러옵니다. 처음 한 번 약 1GB를 내려받습니다"""
    import embed
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from transformers.utils import logging as transformers_logging
    name = 'Qwen/Qwen2.5-0.5B-Instruct'
    embed.ask_endpoint(name)                     # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
    transformers_logging.disable_progress_bar()
    return AutoTokenizer.from_pretrained(name), AutoModelForCausalLM.from_pretrained(name).eval()


print(f'문의 {len(inquiries)}건, 라벨 {LABELS}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 6-1. 프롬프트로만 요구할 때와 구조화 출력

교재의 절: "구조화 출력"

판단하기 어려운 문의 5건을 같은 프롬프트로 두 번씩 보냅니다. 한 번은 프롬프트에 형식만 적고, 한 번은 `schema=`로 구조화 출력을 요청합니다. 두 응답을 각각 `json.loads()`로 파싱해 보고, 어느 쪽에서 파싱 오류나 목록 밖의 라벨이 나오는지 봅니다. 이 셀은 API를 10번 호출합니다.

In [ ]:
SCHEMA = {'type': 'object',
          'properties': {'label': {'type': 'string', 'enum': LABELS}, 'urgent': {'type': 'boolean'}},
          'required': ['label', 'urgent'], 'additionalProperties': False}
ask_format = ('이 문의를 분류하고 판단 이유를 짧게 설명한 뒤, {"label": 라벨, "urgent": 긴급 여부} 형식의 JSON으로 답하세요. '
              f'라벨은 {", ".join(LABELS)} 중 하나입니다.\n문의: ')
for item in inquiries[40:45]:                       # Q41~Q45
    prompt_only = llm.ask(ask_format + item['문의'], model='소형', max_tokens=2000)
    structured = llm.ask(ask_format + item['문의'], model='소형', max_tokens=2000, schema=SCHEMA)
    print(f"{item['번호']} 정답 {item['라벨']}")
    print(f'  프롬프트로만: {parsed(prompt_only.text)}')
    print(f'  구조화 출력: {parsed(structured.text)}')

## 예제 6-2. 다음 토큰 후보 제한하기

교재의 절: "형식을 보장하는 원리"

1장의 공개 LLM에 문의 하나를 분류하라고 하고, 응답이 `{"label": "`까지 생성됐다고 가정합니다. 제약 없이 본 다음 토큰의 상위 5개와, 네 라벨의 첫 토큰만 남기고 다시 계산한 확률을 비교합니다. 상위 후보에 라벨이 아닌 토큰이 있는지, 제약 뒤에 어느 라벨이 1위인지 봅니다. API를 호출하지 않습니다.

In [ ]:
import torch
tokenizer, model = load_local_model()               # 1장의 Qwen2.5-0.5B-Instruct
inquiry = inquiries[42]                             # Q43 냉각수 누수(정답 라벨: 안전)
messages = [{'role': 'user', 'content': f'문의를 {", ".join(LABELS)} 중 하나로 분류해 JSON으로 답하세요.\n문의: {inquiry["문의"]}'}]
prefix = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + '{"label": "'
with torch.no_grad():
    logits = model(**tokenizer(prefix, return_tensors='pt')).logits[0, -1]
probs, top = torch.softmax(logits, dim=-1).topk(5)
print('제약 없음:', ', '.join(f'{tokenizer.decode(token)!r} {prob:.1%}' for prob, token in zip(probs, top)))
allowed = {tokenizer.encode(label)[0]: label for label in LABELS}   # 라벨마다 첫 토큰만 허용합니다
masked = torch.full_like(logits, float('-inf'))
masked[list(allowed)] = logits[list(allowed)]                        # 나머지 토큰의 확률은 0이 됩니다
for token, prob in zip(allowed, torch.softmax(masked, dim=-1)[list(allowed)]):
    print(f'제약 뒤: {tokenizer.decode(token)!r}({allowed[token]}) {prob:.1%}')

## 예제 6-3. 형식은 맞지만 틀린 라벨

교재의 절: "구조화 출력이 보장하지 않는 것"

판단하기 어려운 문의 15건(Q41~Q55)을 배열 스키마의 구조화 출력으로 한 요청에 보내고, 같은 요청을 한 번 더 보냅니다. 형식이 맞은 건수, 정답 라벨과 비교한 정확도, 업무 규칙(안전인데 긴급 아님)에 걸린 건, 두 실행에서 라벨이 다른 건을 봅니다. 이 셀은 API를 2번 호출합니다.

In [ ]:
hard = inquiries[40:55]
ITEMS = {'type': 'object', 'additionalProperties': False, 'required': ['items'], 'properties': {'items': {'type': 'array', 'items': {
    'type': 'object', 'additionalProperties': False, 'required': ['id', 'label', 'urgent'],
    'properties': {'id': {'type': 'string'}, 'label': {'type': 'string', 'enum': LABELS}, 'urgent': {'type': 'boolean'}}}}}}
prompt = '각 문의를 분류하고 긴급 여부를 판단하세요.\n' + '\n'.join(f"{item['번호']}: {item['문의']}" for item in hard)
runs = []
for run in range(2):
    reply = llm.ask(prompt, model='소형', max_tokens=8000, schema=ITEMS)
    runs.append({row['id']: row for row in json.loads(reply.text)['items']})       # 종료 이유가 end일 때만 파싱합니다
    rows = runs[-1]
    correct = sum(rows.get(item['번호'], {}).get('label') == item['라벨'] for item in hard)
    rule = [key for key, row in rows.items() if row['label'] == '안전' and not row['urgent']]
    print(f'{run + 1}회차: 종료 이유 {reply.stop_reason}, 형식이 맞은 건 {len(rows)}/{len(hard)}, '
          f'정확도 {correct}/{len(hard)}, 업무 규칙에 걸린 건 {rule}')
changed = [key for key in runs[0] if runs[0][key]['label'] != runs[1].get(key, {}).get('label')]
print('두 실행에서 라벨이 다른 건:', changed)

## 실습 6-1. 없는 값은 null로

교재의 절: "개념 확인 실습"

문의에서 설비 번호와 현상을 뽑는 스키마 `EXTRACT_SCHEMA`를 작성합니다. 설비 번호(`equipment_id`)는 문의에 없을 수 있으므로 문자열 또는 `null`을 허용하고, 현상(`symptom`)은 문자열로 둡니다. 두 필드 모두 필수입니다. 문의 5건 가운데 `equipment_id`가 `null`로 나올 건수를 먼저 `my_guess`에 적고 실행합니다. 결과에서 뽑은 번호가 문의에 그대로 들어 있는지 확인하고, 스키마가 형식만 보장한다는 것을 이 결과로 설명하면 끝납니다. 이 셀은 API를 5번 호출합니다.

In [ ]:
EXTRACT_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['equipment_id', 'symptom'],
                  'properties': {'equipment_id': {'type': ['string', 'null']}, 'symptom': {'type': 'string'}}}

my_guess = 3
for item in [item for item in inquiries if item['번호'] in ('Q01', 'Q41', 'Q56', 'Q57', 'Q60')]:
    reply = llm.ask('문의에서 설비 번호와 현상을 뽑아 주세요. 설비 번호가 문의에 없으면 null로 둡니다.\n문의: ' + item['문의'],
                    model='소형', max_tokens=2000, schema=EXTRACT_SCHEMA)
    data = json.loads(reply.text)
    found = data['equipment_id'] is None or data['equipment_id'] in item['문의']   # 뽑은 번호가 문의에 그대로 있는지
    print(f"{item['번호']} {item['문의']}\n  → {data}, 문의에 있는 번호인지: {found}")
print('예측:', my_guess)